# Air Quality Forecasting (UCI Air Quality Dataset)
**Coding Ninjas 10X - AI/ML Recruitment, Second Years - Task 1**

**Goal:** predict the CO concentration **one hour ahead** from historical pollutant, sensor and weather readings.

Pipeline: understand & clean data -> EDA -> feature engineering -> chronological split -> models -> error analysis -> insights.

In [ ]:
import os, io, zipfile, urllib.request, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)
SEED = 42

## Part A - Data Understanding & Preprocessing
The file is semicolon-separated, uses a **decimal comma**, and encodes missing values as **-200**.

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/360/air+quality.zip"
if not os.path.exists("AirQualityUCI.csv"):
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read()))
    z.extract("AirQualityUCI.csv")   # if download fails, upload AirQualityUCI.csv manually

raw = pd.read_csv("AirQualityUCI.csv", sep=";", decimal=",")
raw = raw.dropna(axis=1, how="all").dropna(how="all")   # trailing empty columns / rows in the file
print("Rows, columns:", raw.shape)
raw.head()

In [ ]:
raw.info()
raw.describe().T

**Column meaning:** `CO(GT)`, `C6H6(GT)`, `NOx(GT)`, `NO2(GT)` = reference-analyser ground truth; `PT08.S1..S5` = low-cost metal-oxide sensor responses; `T`, `RH`, `AH` = temperature, relative & absolute humidity; `NMHC(GT)` = non-methane hydrocarbons.

In [ ]:
# Missing values are stored as -200 in this dataset
print("Count of -200 per column:")
print((raw.select_dtypes("number") == -200).sum())

In [ ]:
# Time information: build a proper datetime index
df = raw.copy()
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"].str.replace(".", ":", regex=False),
                                format="%d/%m/%Y %H:%M:%S")
df = df.drop(columns=["Date", "Time"]).set_index("datetime").sort_index()

print("Period:", df.index.min(), "->", df.index.max())
print("Duplicate timestamps:", df.index.duplicated().sum())
print("Time step between rows:"); print(df.index.to_series().diff().value_counts().head())
full_idx = pd.date_range(df.index.min(), df.index.max(), freq="h")
print("Missing hourly timestamps:", len(full_idx.difference(df.index)))

In [ ]:
# Clean: rename, -200 -> NaN, complete hourly index (so shift(24) really means 24 hours ago)
df = df.rename(columns={"CO(GT)":"CO","PT08.S1(CO)":"S1_CO","NMHC(GT)":"NMHC","C6H6(GT)":"C6H6",
                        "PT08.S2(NMHC)":"S2_NMHC","NOx(GT)":"NOx","PT08.S3(NOx)":"S3_NOx",
                        "NO2(GT)":"NO2","PT08.S4(NO2)":"S4_NO2","PT08.S5(O3)":"S5_O3"})
df = df.replace(-200, np.nan).asfreq("h")

miss = df.isna().mean().mul(100).sort_values(ascending=False)
miss.plot.bar(color="tomato", title="% missing per column (after -200 -> NaN)"); plt.ylabel("%"); plt.show()
print(miss.round(1))

**Missing-value strategy**
- `NMHC` is ~90% missing -> **dropped** (imputing it would be inventing data).
- Short gaps (<= 3 hours) are filled with a **forward fill** (last known value). Forward fill only uses the *past*, so it does not leak future information (unlike interpolation or backward fill).
- Longer gaps stay NaN; rows that still have NaN are dropped later (after lag features are built, so lags stay correct).

In [ ]:
df = df.drop(columns=["NMHC"])
df = df.ffill(limit=3)

# Invalid / unusual value checks
print("Negative values:", (df.select_dtypes("number") < 0).sum().sum() - (df[["T"]] < 0).sum().sum(), "(T can legitimately be < 0)")
print("RH outside 0-100:", ((df.RH < 0) | (df.RH > 100)).sum())
print("Remaining missing per column:"); print(df.isna().sum())
df.describe().T.round(2)

## Part B - Exploratory Data Analysis

In [ ]:
cols = ["CO", "C6H6", "NOx", "NO2"]
fig, ax = plt.subplots(1, 4, figsize=(16, 3.5))
for a, c in zip(ax, cols):
    sns.histplot(df[c].dropna(), kde=True, ax=a); a.set_title(f"Distribution of {c}")
plt.tight_layout(); plt.show()
print(df[cols].skew().round(2).rename("skewness"))

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 8})
plt.title("Correlation matrix"); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, c in zip(ax, ["C6H6", "NOx", "T"]):
    a.scatter(df[c], df["CO"], s=4, alpha=.3); a.set_xlabel(c); a.set_ylabel("CO")
    a.set_title(f"CO vs {c}")
plt.tight_layout(); plt.show()

In [ ]:
# Air quality over time
fig, ax = plt.subplots(figsize=(13, 4))
df["CO"].plot(ax=ax, alpha=.35, label="hourly")
df["CO"].rolling(24*7, min_periods=24).mean().plot(ax=ax, color="red", label="7-day mean")
ax.set_title("CO over time"); ax.legend(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
df.groupby(df.index.hour)["CO"].mean().plot(ax=ax[0], marker="o", title="Mean CO by hour of day")
df.groupby(df.index.dayofweek)["CO"].mean().plot(ax=ax[1], marker="o", title="Mean CO by day of week (0=Mon)")
df.groupby(df.index.to_period("M"))["CO"].mean().plot(ax=ax[2], marker="o", title="Mean CO by month")
plt.tight_layout(); plt.show()

In [ ]:
# Unusual observations: z-score of CO
z = (df["CO"] - df["CO"].mean()) / df["CO"].std()
print("Hours with |z| > 3:", (z.abs() > 3).sum())
print(df.loc[z.abs() > 3, ["CO", "C6H6", "NOx", "NO2"]].sort_values("CO", ascending=False).head(8))

**EDA takeaways:** CO is right-skewed (occasional high spikes); it follows a clear **daily cycle** (morning and evening rush-hour peaks, lower at night), differs on weekends, and is strongly correlated with C6H6, NOx and the S1/S2 sensors. Compare these statements with your own plots above.

## Part C - Feature Engineering
| Feature | Why it helps |
|---|---|
| **Lags** of CO (1, 2, 3, 6, 24 h) | Pollution is persistent - the last hours strongly predict the next; lag 24 captures the daily cycle |
| **Rolling mean / std** of CO (3, 6, 24 h) | Smooths noise, shows recent trend and volatility |
| **Hour (sin/cos), day of week, weekend flag, month** | Traffic and season patterns; sin/cos lets the model know 23:00 is next to 00:00 |
| Current C6H6, NOx, NO2, sensors, T, RH, AH | Co-pollutants and weather conditions right now |

**Target:** `CO` at the **next hour** (t+1).

In [ ]:
feat = df.copy()
for lag in [1, 2, 3, 6, 24]:
    feat[f"CO_lag{lag}"] = df["CO"].shift(lag)
for w in [3, 6, 24]:
    feat[f"CO_roll{w}_mean"] = df["CO"].rolling(w, min_periods=w//2).mean()   # windows end at time t (past only)
feat["CO_roll6_std"] = df["CO"].rolling(6, min_periods=3).std()

feat["hour"] = feat.index.hour
feat["hour_sin"] = np.sin(2*np.pi*feat["hour"]/24)
feat["hour_cos"] = np.cos(2*np.pi*feat["hour"]/24)
feat["dayofweek"] = feat.index.dayofweek
feat["is_weekend"] = (feat["dayofweek"] >= 5).astype(int)
feat["month"] = feat.index.month
feat = feat.drop(columns="hour")

feat["target_CO_next_hour"] = df["CO"].shift(-1)

before = len(feat)
feat = feat.dropna()
print(f"Rows: {before} -> {len(feat)} after dropping rows with missing features/target")
X = feat.drop(columns="target_CO_next_hour"); y = feat["target_CO_next_hour"]
print("Features:", X.shape[1]); list(X.columns)

## Part D - Prediction
**Chronological split:** first 80% of time for training, last 20% for testing - *no shuffling*, so the model never trains on data from after the test period.

In [ ]:
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]
print("Train:", X_train.index.min(), "->", X_train.index.max(), len(X_train))
print("Test :", X_test.index.min(), "->", X_test.index.max(), len(X_test))

def metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"MAE": mean_absolute_error(y_true, y_pred), "MSE": mse, "RMSE": np.sqrt(mse), "R2": r2_score(y_true, y_pred)}

models = {
    "Ridge (linear)": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),   # scaler is fit on train only
    "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=SEED),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=SEED),
}

results = {"Baseline: CO(t) persistence": metrics(y_test, X_test["CO"])}   # 'next hour = this hour'
preds = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    preds[name] = m.predict(X_test)
    results[name] = metrics(y_test, preds[name])
    results[name + " (train)"] = metrics(y_train, m.predict(X_train))

res = pd.DataFrame(results).T.round(3)
res

In [ ]:
test_res = res.loc[[i for i in res.index if "(train)" not in i]]
print(test_res)
best_name = test_res.drop(index="Baseline: CO(t) persistence")["RMSE"].idxmin()
best = models[best_name]; best_pred = preds[best_name]
print("\nBest ML model on test RMSE:", best_name)

## Part E - Analysis

In [ ]:
# Predictions vs actual
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [2, 1]})
ser = pd.DataFrame({"actual": y_test, "pred": best_pred}, index=y_test.index)
ser.iloc[:24*14].plot(ax=ax[0], title=f"Actual vs predicted next-hour CO (first 2 test weeks) - {best_name}")
ax[1].scatter(ser.actual, ser.pred, s=5, alpha=.4)
lim = [0, ser.actual.max()]; ax[1].plot(lim, lim, "r--"); ax[1].set_xlabel("Actual"); ax[1].set_ylabel("Predicted")
ax[1].set_title("Actual vs predicted (test)"); plt.tight_layout(); plt.show()

In [ ]:
# Where does it do well / badly?
ser["abs_err"] = (ser.actual - ser.pred).abs()
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ser.groupby(ser.index.hour)["abs_err"].mean().plot(ax=ax[0], marker="o", title="MAE by hour of day")
ser.groupby(pd.qcut(ser.actual, 5, duplicates="drop"))["abs_err"].mean().plot.bar(ax=ax[1], title="MAE by actual CO level (quintile)")
ser.groupby(ser.index.to_period("M"))["abs_err"].mean().plot(ax=ax[2], marker="o", title="MAE by month")
plt.tight_layout(); plt.show()
print("Largest errors:"); print(ser.sort_values("abs_err", ascending=False).head(8).round(2))

In [ ]:
# Feature importance (permutation importance on the test set)
pi = permutation_importance(best, X_test, y_test, n_repeats=5, random_state=SEED, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False)
imp.head(12)[::-1].plot.barh(title="Top features (permutation importance, drop in R2)"); plt.show()
print(imp.head(10).round(4))

### Data leakage in time-series prediction
**Issue:** leakage happens when the model sees information at training time that would not be available at prediction time. Typical sources here:
1. **Random train/test split** - training on Nov data and testing on Oct lets the model "peek at the future". *Avoided:* chronological split (train = past, test = future).
2. **Features that use future values** - centered rolling windows, backward fill, or interpolation across a gap use data from *after* time t. *Avoided:* rolling windows end at t, lags are `shift(+k)`, and gaps are filled with forward fill only.
3. **Target in the features** - the target is `CO(t+1)` via `shift(-1)`; every feature is known at time t. Scaling (Ridge pipeline) is fitted on the training set only.

## Part F - Insights

In [ ]:
b = res.loc["Baseline: CO(t) persistence"]; t = res.loc[best_name]; tr = res.loc[best_name + " (train)"]
print(f"1. Best model: {best_name} -> test MAE {t.MAE:.3f}, RMSE {t.RMSE:.3f}, R2 {t.R2:.3f}")
print(f"2. Persistence baseline: MAE {b.MAE:.3f}, RMSE {b.RMSE:.3f}, R2 {b.R2:.3f}")
print(f"3. Train R2 {tr.R2:.3f} vs test R2 {t.R2:.3f}  (gap = {tr.R2 - t.R2:.3f})")
print("4. Top features:", ", ".join(imp.head(3).index))
print("5. Worst hours of day (MAE):", ", ".join(map(str, ser.groupby(ser.index.hour)['abs_err'].mean().nlargest(3).index)))

**Key findings** *(check each against the outputs above and edit with your own numbers)*
1. Data is hourly (Mar 2004 - Apr 2005); missing values are encoded as -200 and `NMHC` is ~90% missing, so it was dropped.
2. CO has a strong **daily rhythm** (rush-hour peaks) and a weekday/weekend difference, and correlates strongly with C6H6, NOx and the S1/S2 sensors.
3. CO is highly **persistent**: recent CO values (lag 1, rolling means) are the most important predictors.
4. The persistence baseline is already strong; the ML model improves on it only modestly - a good reminder to always compare against a simple baseline.
5. Errors are largest at **peak/spike hours** and high CO levels, which are hard to anticipate one hour ahead.
6. Train vs test gap shows how much the model overfits (see the printed R2 gap).

**Limitations**
1. The test set covers a different **season** (winter -> spring) than most of the training data, so it is a distribution shift; the dataset is also only ~1 year long and from a single location.
2. Many rows were lost to missing values/lag construction, and the sensors drift over time.
3. Single-step (1 hour) forecast only; spikes are under-predicted by tree models that average.

**Improvement / next steps:** tune hyperparameters with `TimeSeriesSplit`, try multi-step forecasting and sequence models (LSTM/GRU), and add more history (a full year+ of training data to see every season).